In [ ]:
"""
Outlook Email Categorizer
=========================
Reads emails from Outlook for the past N days, categorizes them by subject keywords,
assigns priority levels, and outputs a formatted DataFrame (and optional Excel file).

Uses win32com.client to talk directly to your locally installed Outlook —
no OAuth, no API keys, no browser login needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ONE-TIME SETUP (do this before running the script)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

STEP 1 — Install required libraries:
    pip install pywin32 pandas openpyxl

STEP 2 — Make sure Outlook is open and running on your machine.
    win32com connects to the running Outlook process, so it must be open.

STEP 3 — Configure the settings below, then run.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NOTE ON OUTLOOK RULES & SUBFOLDERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
If you have Outlook rules that route emails into subfolders, those emails will
NOT appear in the Inbox. This script handles that in two ways:

  Option A (default) — Set SEARCH_SUBFOLDERS = True
      The script will walk the Inbox AND all subfolders recursively.
      This catches everything regardless of where rules moved it.

  Option B — Set SEARCH_SUBFOLDERS = False and list specific folders in
      FOLDER_NAMES below (e.g. ['Inbox', 'Alerts', 'Team Emails']).
      Use this if you only care about specific known folders.
"""


In [ ]:
import re
import pandas as pd
from datetime import datetime, timedelta

import win32com.client


# ──────────────────────────────────────────────
# CONFIGURATION — Edit these rules as needed
# ──────────────────────────────────────────────

CATEGORY_RULES = [
    {
        "category": "Cancelled",
        "priority": 1,
        # Any of these patterns (case-insensitive) will match
        "patterns": [
            r"cancel",      # catches: cancelled, cancellation, canceling, etc.
        ],
    },
    {
        "category": "Comm Delay",
        "priority": 2,
        "patterns": [
            r"comm(unication)?\s*(delay|delays|delayed)",
            r"communication\s*delay",
        ],
    },
    # ── Add more categories below as needed ──
    # {
    #     "category": "Outage",
    #     "priority": 3,
    #     "patterns": [r"outage", r"power\s*out"],
    # },
]

# How many days back to search
DAYS_BACK = 30

# Set to True to also save the result as an Excel file
SAVE_TO_EXCEL = True
EXCEL_OUTPUT_PATH = "email_categories.xlsx"

# ── Folder settings ──
# True  → search Inbox + ALL subfolders recursively (handles Outlook rules)
# False → only search the folders listed in FOLDER_NAMES
SEARCH_SUBFOLDERS = True

# Only used when SEARCH_SUBFOLDERS = False.
# List folder names exactly as they appear in Outlook.
# 'Inbox' is always included automatically.
FOLDER_NAMES = ["Inbox"]


In [ ]:
# ──────────────────────────────────────────────
# HELPER FUNCTIONS
# ──────────────────────────────────────────────

def get_outlook_inbox():
    """
    Connects to the running Outlook instance and returns the default Inbox folder.
    Outlook must be open on your machine for this to work.
    """
    outlook = win32com.client.Dispatch("Outlook.Application")
    namespace = outlook.GetNamespace("MAPI")
    # 6 = olFolderInbox
    inbox = namespace.GetDefaultFolder(6)
    return inbox


def get_all_folders(folder, collected=None):
    """
    Recursively walks a folder and all its subfolders.
    Returns a flat list of all folder objects found.
    """
    if collected is None:
        collected = []
    collected.append(folder)
    for subfolder in folder.Folders:
        get_all_folders(subfolder, collected)
    return collected


def categorize_subject(subject: str):
    """
    Returns (priority, category) if the subject matches a rule, else (None, None).
    Rules are checked in order — first match wins.
    """
    for rule in CATEGORY_RULES:
        for pattern in rule["patterns"]:
            if re.search(pattern, subject, re.IGNORECASE):
                return rule["priority"], rule["category"]
    return None, None


def format_dates(dt: datetime):
    """Returns (date_sent, month_sent) formatted strings."""
    if dt is None:
        return "Unknown", "Unknown"
    date_sent = f"{dt.month}/{dt.day}/{str(dt.year)[2:]}"   # e.g. 2/12/26
    month_sent = dt.strftime("%b-%y")                        # e.g. Feb-26
    return date_sent, month_sent


def search_folder(folder, cutoff_dt, rows):
    """
    Searches a single Outlook folder for emails after cutoff_dt that match
    a category rule. Appends matching rows to the rows list.

    Uses Outlook's built-in DASL filter for efficiency — only fetches emails
    newer than the cutoff, rather than looping over everything.
    """
    # Outlook DASL filter — much faster than looping all items
    # Format: MM/DD/YYYY HH:MM AM/PM
    cutoff_str = cutoff_dt.strftime("%m/%d/%Y %I:%M %p")
    filter_str = f"[ReceivedTime] >= '{cutoff_str}'"

    try:
        items = folder.Items
        items.Sort("[ReceivedTime]", True)  # newest first
        filtered = items.Restrict(filter_str)
    except Exception as e:
        print(f"  Could not search folder '{folder.Name}': {e}")
        return

    for item in filtered:
        try:
            # Only process mail items (not calendar invites, etc.)
            if item.Class != 43:  # 43 = olMail
                continue

            subject = item.Subject or "(No Subject)"
            priority, category = categorize_subject(subject)

            if priority is None:
                continue

            received = item.ReceivedTime
            # win32com returns a pywintypes.datetime — convert to standard datetime
            dt = datetime(received.year, received.month, received.day,
                          received.hour, received.minute, received.second)
            date_sent, month_sent = format_dates(dt)

            rows.append({
                "Priority":           priority,
                "Category":           category,
                "Header Name":        subject,
                "Folder":             folder.Name,
                "Date Sent mm-dd-yy": date_sent,
                "Month Sent mm-yy":   month_sent,
            })

        except Exception as e:
            print(f"  Skipped a message due to error: {e}")
            continue


# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────

def main():
    print("Connecting to Outlook...")
    inbox = get_outlook_inbox()
    print("Connected!\n")

    cutoff_dt = datetime.now() - timedelta(days=DAYS_BACK)

    # Build the list of folders to search
    if SEARCH_SUBFOLDERS:
        folders = get_all_folders(inbox)
        print(f"Searching Inbox + all subfolders ({len(folders)} folders total)...")
    else:
        # Get the parent store to look up named folders
        outlook = win32com.client.Dispatch("Outlook.Application")
        namespace = outlook.GetNamespace("MAPI")
        folders = [inbox]
        for name in FOLDER_NAMES:
            if name.lower() == "inbox":
                continue  # already added
            try:
                folder = inbox.Parent.Folders[name]
                folders.append(folder)
            except Exception:
                print(f"  Warning: Could not find folder '{name}' — skipping.")
        print(f"Searching {len(folders)} specified folder(s)...")

    print(f"Date range: past {DAYS_BACK} days (since {cutoff_dt.strftime('%Y-%m-%d')})\n")

    rows = []
    for folder in folders:
        search_folder(folder, cutoff_dt, rows)

    if not rows:
        print("No emails matched any of the configured categories.")
        return

    # Build DataFrame and sort by Priority then Date
    df = pd.DataFrame(rows, columns=[
        "Priority", "Category", "Header Name", "Folder",
        "Date Sent mm-dd-yy", "Month Sent mm-yy"
    ])
    df.sort_values(by=["Priority", "Date Sent mm-dd-yy"], ascending=[True, False], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Print to console
    print(f"\n{'='*80}")
    print(f"  EMAIL CATEGORY REPORT  |  Past {DAYS_BACK} days  |  {len(df)} matches")
    print(f"{'='*80}")
    print(df.to_string(index=False))
    print(f"{'='*80}\n")

    # Optionally save to Excel
    if SAVE_TO_EXCEL:
        with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine="openpyxl") as writer:
            df.to_excel(writer, index=False, sheet_name="Email Categories")

            # Auto-fit column widths
            ws = writer.sheets["Email Categories"]
            for col in ws.columns:
                max_len = max(len(str(cell.value or "")) for cell in col)
                ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 60)

        print(f"Results saved to: {EXCEL_OUTPUT_PATH}")


if __name__ == "__main__":
    main()


main()
